[문제 정의]

- 단순한 최종 성적 비교를 넘어, "어떤 요인이 성적의 상승과 하락을 이끄는가?"를 파악하고자 함.

- 전공별로 성적에 영향을 미치는 주요 생활 습관(수면, 교우 시간 등)이 다를 것이라는 가설 설정.

[데이터셋 선택]

- 학생들의 학업 성취도와 생활 습관 데이터(Student_data.csv)를 기본으로 활용.

- (향후 계획) 필요시 전공별 특성을 보완할 외부 데이터 결합 예정.

In [28]:
import pandas as pd

df = pd.read_csv('../data/Student_data.csv')

display(df.head())

,Student_ID,Gender,Age,Major,Attendance_Pct,Study_Hours_Per_Day,Previous_CGPA,Sleep_Hours,Social_Hours_Week,Final_CGPA
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34


In [30]:

# rename columns
df.rename(columns={
    'Student_ID': 'ID',
    'Attendance_Pct': 'Attendance',
    'Study_Hours_Per_Day': 'Study',
    'Previous_CGPA': 'P_CGPA',
    'Sleep_Hours': 'Sleep',
    'Social_Hours_Week': 'Social_Hours',
    'Final_CGPA': 'F_CGPA',
}, inplace=True)

display(df.head())

,ID,Gender,Age,Major,Attendance,Study,P_CGPA,Sleep,Social_Hours,F_CGPA
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34


In [23]:
# 결측치 확인
print("=== 누락 데이터(결측치) 확인 ===")
print(df.isnull().sum()) 

# 데이터 타입 확인
print("\n=== 각 데이터의 타입 확인 ===")
print(df.dtypes) 
print("-" * 50)

=== 누락 데이터(결측치) 확인 ===
ID              0
Gender          0
Age             0
Major           0
Attendance      0
Study           0
P_CGPA          0
Sleep           0
Social_Hours    0
F_CGPA          0
dtype: int64

=== 각 데이터의 타입 확인 ===
ID                  str
Gender              str
Age               int64
Major               str
Attendance      float64
Study           float64
P_CGPA          float64
Sleep           float64
Social_Hours      int64
F_CGPA          float64
dtype: object
--------------------------------------------------


# 데이터 다듬기

In [ ]:

# 상위 15% 여부 판단 컬럼 추가
top_15_threshold = df['F_CGPA'].quantile(0.85)
df['Top_15'] = df['F_CGPA'] >= top_15_threshold

display(df.head())

,ID,Gender,Age,Major,Attendance,Study,P_CGPA,Sleep,Social_Hours,F_CGPA,Top_15
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78,False
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76,False
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75,False
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69,False
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34,False


In [34]:

# 성적 편차 컬럼 추가
df['CGPA_Diff'] = df['F_CGPA'] - df['P_CGPA']

def categorize_trend(diff):
    if abs(diff) <= 0.1:
        return 'Maintain'   # 성적 유지
    elif diff <= -0.5:
        return 'Decline'    # 성적 하락
    elif diff >= 0.5:
        return 'Improve'    # 성적 향상
    else:
        return 'Other'      # 기타

df['Trend'] = df['CGPA_Diff'].apply(categorize_trend)

display(df.head())

,ID,Gender,Age,Major,Attendance,Study,P_CGPA,Sleep,Social_Hours,F_CGPA,Top_15,CGPA_Diff,Trend
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78,False,0.13,Other
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76,False,0.18,Other
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75,False,0.46,Other
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69,False,0.21,Other
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34,False,-0.32,Other


In [36]:
# 최종 성적 대비 공부시간 효율성 컬럼 추가
df['Efficiency'] = round(df['F_CGPA'] / df['Study'], 2)

display(df.head())

,ID,Gender,Age,Major,Attendance,Study,P_CGPA,Sleep,Social_Hours,F_CGPA,Top_15,CGPA_Diff,Trend,Efficiency
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78,False,0.13,Other,0.63
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76,False,0.18,Other,0.94
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75,False,0.46,Other,0.96
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69,False,0.21,Other,0.42
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34,False,-0.32,Other,1.06


In [37]:
bins = [0, 75, 90, 100]
labels = ['Danger', 'Normal', 'Excellent']
df['Attendance_Grade'] = pd.cut(df['Attendance'], bins=bins, labels=labels)

display(df.head())

,ID,Gender,Age,Major,Attendance,Study,P_CGPA,Sleep,Social_Hours,F_CGPA,Top_15,CGPA_Diff,Trend,Efficiency,Attendance_Grade
0,ID00001,Male,20,Engineering,83.9,4.4,2.65,9.1,8,2.78,False,0.13,Other,0.63,Normal
1,ID00002,Female,24,Business,80.7,4.0,3.58,4.0,4,3.76,False,0.18,Other,0.94,Normal
2,ID00003,Female,20,Mathematics,91.5,3.9,3.29,6.7,4,3.75,False,0.46,Other,0.96,Excellent
3,ID00004,Female,23,Engineering,73.9,8.8,3.48,4.0,6,3.69,False,0.21,Other,0.42,Danger
4,ID00005,Male,21,Economics,79.8,2.2,2.66,8.7,6,2.34,False,-0.32,Other,1.06,Normal


In [39]:
# ==========================================
# [데이터 가독성을 위한 컬럼 순서 재배치]
# ==========================================
# 보기 편한 논리적 흐름(학생정보 -> 생활습관 -> 성적결과)으로 순서 정의

new_column_order = [
    # Demographics
    'ID', 'Gender', 'Age', 'Major',
    # Lifestyle & Time Management & Inputs
    'Sleep', 'Study', 'Social_Hours', 'Attendance', 'Attendance_Grade',
    # Academic Performance & Outputs
    'P_CGPA', 'F_CGPA', 'CGPA_Diff', 'Trend', 'Efficiency', 'Top_15'
]

df = df[new_column_order]

# ==========================================
# [최종 데이터 저장 및 출력]
# ==========================================
# 잘 정돈된 데이터를 최종 저장합니다.

df.to_csv('Student_data_Final.csv', index=False)

display(df.head())

,ID,Gender,Age,Major,Sleep,Study,Social_Hours,Attendance,Attendance_Grade,P_CGPA,F_CGPA,CGPA_Diff,Trend,Efficiency,Top_15
0,ID00001,Male,20,Engineering,9.1,4.4,8,83.9,Normal,2.65,2.78,0.13,Other,0.63,False
1,ID00002,Female,24,Business,4.0,4.0,4,80.7,Normal,3.58,3.76,0.18,Other,0.94,False
2,ID00003,Female,20,Mathematics,6.7,3.9,4,91.5,Excellent,3.29,3.75,0.46,Other,0.96,False
3,ID00004,Female,23,Engineering,4.0,8.8,6,73.9,Danger,3.48,3.69,0.21,Other,0.42,False
4,ID00005,Male,21,Economics,8.7,2.2,6,79.8,Normal,2.66,2.34,-0.32,Other,1.06,False


In [35]:
# 가공해서 새로 저장 - 필요시 주석 해제
df.to_csv('Student_data_processed.csv', index=False, encoding='utf-8-sig')

# 결과 출력

In [38]:
# 상위 15퍼 성적
print(f"\n[기준] 전체 학생 기준 상위 15% 최종 성적 커트라인: {top_15_threshold:.2f}점\n")

# 1. '성적 유지' 그룹에서 꾸준히 성적이 높은(상위 15%) 사람 목록
maintain_high = df[(df['Trend'] == 'Maintain') & (df['Top_15'] == True)][['Study', 'Sleep', 'Attendance', 'Social_Hours']]
print(f"=== 1-1. 성적 유지 & 꾸준히 높은 학생 목록 (총 {len(maintain_high)}명) ===")
display(maintain_high.head())

# 2. '성적 하락' 그룹 목록
decline = df[df['Trend'] == 'Decline'][['Study', 'Sleep', 'Attendance', 'Social_Hours']]
print(f"\n=== 1-2. 성적 하락 학생 목록 (총 {len(decline)}명) ===")
display(decline.head())

# 3. '성적 향상' 그룹 목록
improve = df[df['Trend'] == 'Improve'][['Study', 'Sleep', 'Attendance', 'Social_Hours']]
print(f"\n=== 1-3. 성적 향상 학생 목록 (총 {len(improve)}명) ===")
display(improve.head())

# 4. 각 전공별(데이터상 6개 전공 존재) 최근 성적 상위 15% 학생 목록
print("\n=== 2-1. 각 전공 분야별 성적 상위 15% 학생 특징 ===")
majors = df['Major'].unique()

for m in majors:
    m_df = df[df['Major'] == m]
    # 각 전공별 상위 15% 계산
    m_top_15_thresh = m_df['F_CGPA'].quantile(0.85)
    m_top_15 = m_df[m_df['F_CGPA'] >= m_top_15_thresh][['Study', 'Sleep', 'Attendance', 'Social_Hours']]
    
    print(f"\n[{m}] 전공 상위 15% (커트라인: {m_top_15_thresh:.2f}점, 인원: {len(m_top_15)}명)")
    display(m_top_15.head())


[기준] 전체 학생 기준 상위 15% 최종 성적 커트라인: 3.87점

=== 1-1. 성적 유지 & 꾸준히 높은 학생 목록 (총 253명) ===


,Study,Sleep,Attendance,Social_Hours
8,4.3,5.0,100.0,12
10,2.2,6.8,60.0,14
115,6.8,8.8,79.6,8
132,4.0,7.1,97.3,9
137,4.6,7.9,81.7,6



=== 1-2. 성적 하락 학생 목록 (총 35명) ===


,Study,Sleep,Attendance,Social_Hours
278,4.1,5.7,52.1,11
344,2.2,9.6,59.4,4
363,2.7,9.0,59.4,9
386,2.3,4.5,58.5,6
425,2.3,8.1,56.1,11



=== 1-3. 성적 향상 학생 목록 (총 425명) ===


...

,Study,Sleep,Attendance,Social_Hours
70,4.7,7.0,81.7,2
120,5.9,4.0,88.9,9
204,8.6,6.9,100.0,10
309,2.9,6.8,88.5,10
358,6.3,7.7,100.0,13



[Psychology] 전공 상위 15% (커트라인: 3.91점, 인원: 135명)


,Study,Sleep,Attendance,Social_Hours
44,3.9,6.2,90.1,4
85,3.5,6.4,80.5,8
124,12.5,8.5,79.6,11
232,2.3,7.9,100.0,3
245,2.3,6.4,100.0,9



[Computer Science] 전공 상위 15% (커트라인: 3.90점, 인원: 126명)


,Study,Sleep,Attendance,Social_Hours
8,4.3,5.0,100.0,12
47,4.5,9.1,81.0,5
96,1.0,5.1,92.1,8
137,4.6,7.9,81.7,6
185,5.2,7.2,100.0,6


# 데이터 저장하고 싶으면 원본(Student_data.csv)은 그대로 두고, 
# 가공된 데이터(df)를 'Student_data_processed.csv'라는 새 이름으로 저장하기
# df.to_csv('../data/Student_data_processed.csv', index=False)